# Fusion Manipulation-Risk Notebook

This notebook is the public-facing execution and explanation layer for the Fusion pipeline.

It is designed for two audiences at the same time:

- human reviewers who need a clear story
- developers who need a repeatable execution path

The notebook calls the package implementation in `fusion_pipeline` and does not define an independent scoring system.


## Modeling Philosophy And Limits

Fusion is an **unsupervised manipulation-risk scoring system**.

It does **not** claim that suspicious text alone proves automation.
A human can manually write spam-looking text. Because of that, the strongest evidence in this project is **behavioral context**, especially:

- author posting intensity
- inter-post timing
- repeated text reuse
- multi-author reuse of the same text
- short-window clustering

The semantic model is a supporting signal. It improves ranking when textual cues are suspicious, but the behavioral score remains the primary evidence source.


In [ ]:
BASE_CONFIG = {
    "paths": {
        "input_parquet": "data/datathonFINAL.parquet",
        "clean_output_parquet": "data/datathonFINAL_formula_cleaned.parquet",
        "batch_sqlite_db": "data/fusion_batch_store.sqlite",
        "author_scores_parquet": "data/fusion_author_scores.parquet",
        "scored_messages_parquet": "data/fusion_scored_messages.parquet",
        "manifest_path": "data/fusion_manifest.json",
        "author_feature_stage_parquet": "data/fusion_author_feature_stage.parquet",
    },
    "runtime": {
        "mode": "full",
        "build_mode": "lean",
        "batch_size": 50_000,
        "max_batches": None,
        "sample_n_rows": None,
        "overwrite_outputs": True,
        "enable_progress_logs": True,
        "progress_every_batches": 5,
        "top_n_domain_context": 128,
        "author_batch_size": 5_000,
        "message_batch_size": 100_000,
    },
    "semantic_adapter": {
        "enabled": True,
        "model_name": "junaid1993/distilroberta-bot-detection",
        "supported_languages": ["en"],
        "unsupported_language_score": 0.50,
        "max_length": 512,
        "batch_size": 32,
        "device": "auto",
    },
    "thresholds": {
        "min_text_len": 5,
        "rare_language_threshold": 100,
        "long_text_start": 30,
        "hourly_penalty_start": 10,
        "language_penalty_start": 4,
        "hard_bot_time_window_sec": 300,
        "hard_bot_repeat_threshold": 5,
        "hard_bot_multi_author_threshold": 2,
        "hard_bot_time_cluster_threshold": 3,
        "spam_repeat_threshold": 3,
        "spam_multi_author_threshold": 2,
        "spam_time_cluster_threshold": 3,
    },
    "rules": {
        "long_text_requires_spam": False,
    },
    "dynamic_final_weighting": {
        "enabled": True,
        "min_confidence_weight": 0.20,
        "power": 2.0,
        "sigmoid_steepness": 8.0,
    },
    "neutral_score_policy": {
        "neutral_score": 0.50,
        "epsilon": 1e-6,
    },
    "dominant_signal_policy": {
        "enabled": True,
        "mode": "floor",
        "threshold": 0.68,
        "repeat_hard_threshold": 5,
        "scope": {
            "author": True,
            "message": True,
        },
    },
    "derived_thresholds": {},
    "weights": {
        "behavioral_vs_semantic": {
            "behavioral": 0.60,
            "semantic": 0.40,
        },
        "author_vs_message": {
            "author": 0.70,
            "message": 0.30,
        },
        "author_components": {
            "activity": 0.35,
            "timing": 0.25,
            "repetition": 0.30,
            "diversity": 0.10,
            "metadata": 0.00,
        },
        "message_components": {
            "same_text_repeat": 0.24,
            "spam_pattern": 0.25,
            "hashtag_spam": 0.18,
            "token_repetition": 0.15,
            "long_text": 0.10,
            "keyword_signal": 0.08,
        },
    },
}

BASE_CONFIG

In [ ]:
import copy


def build_preset(base_config: dict, *, mode: str, sample_n_rows=None, max_batches=None, semantic_enabled=True) -> dict:
    config = copy.deepcopy(base_config)
    config["runtime"]["mode"] = mode
    config["runtime"]["sample_n_rows"] = sample_n_rows
    config["runtime"]["max_batches"] = max_batches
    config["semantic_adapter"]["enabled"] = semantic_enabled
    return config


MODEL_TEST_CONFIG = build_preset(
    BASE_CONFIG,
    mode="model_test",
    sample_n_rows=200_000,
    max_batches=2,
    semantic_enabled=True,
)

FULL_BUILD_CONFIG = build_preset(
    BASE_CONFIG,
    mode="full",
    sample_n_rows=None,
    max_batches=None,
    semantic_enabled=True,
)

DEMO_CONFIG = build_preset(
    BASE_CONFIG,
    mode="demo",
    sample_n_rows=5_000,
    max_batches=1,
    semantic_enabled=False,
)

PRESETS = {
    "model_test": MODEL_TEST_CONFIG,
    "full_build": FULL_BUILD_CONFIG,
    "demo": DEMO_CONFIG,
}

ACTIVE_PRESET = "demo"
EXECUTION_MODE = "build"  # build | rescore | validate_only
ACTIVE_CONFIG = copy.deepcopy(PRESETS[ACTIVE_PRESET])

{
    "available_presets": list(PRESETS),
    "active_preset": ACTIVE_PRESET,
    "execution_mode": EXECUTION_MODE,
}


In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from fusion_pipeline import (
    assert_artifacts_ready,
    prepare_single_message_input,
    run_formula_pipeline_two_pass,
    run_rescore_from_existing_store,
    score_single_message,
)
from fusion_pipeline.artifacts import (
    plot_hourly_distribution,
    plot_hourly_penalty_curve,
    summarize_score_bands,
)


def attach_runtime_paths(result: dict, config: dict) -> dict:
    result.setdefault("paths", {})
    result["paths"].setdefault("sqlite_db", config["paths"]["batch_sqlite_db"])
    result["paths"].setdefault("author_scores_parquet", config["paths"]["author_scores_parquet"])
    result["paths"].setdefault("scored_messages_parquet", config["paths"]["scored_messages_parquet"])
    result["paths"].setdefault("manifest_json", config["paths"]["manifest_path"])
    return result


def load_scored_messages(config: dict, columns=None) -> pd.DataFrame:
    path = Path(config["paths"]["scored_messages_parquet"])
    return pd.read_parquet(path, columns=columns)


def load_author_scores(config: dict) -> pd.DataFrame:
    return pd.read_parquet(config["paths"]["author_scores_parquet"])


def load_message_context(config: dict) -> pd.DataFrame:
    conn = sqlite3.connect(config["paths"]["batch_sqlite_db"])
    try:
        return pd.read_sql_query(
            """
            SELECT message_id, language, registered_domain, date
            FROM cleaned_messages
            """,
            conn,
        )
    finally:
        conn.close()


def build_compact_manipulation_map(config: dict, *, min_rows: int = 25) -> pd.DataFrame:
    scored = load_scored_messages(
        config,
        columns=["message_id", "final_score", "hard_bot_cluster_flag", "hard_same_text_repeat_flag"],
    )
    context = load_message_context(config)
    merged = scored.merge(context, on="message_id", how="left")
    grouped = (
        merged.groupby(["language", "registered_domain"], dropna=False)
        .agg(
            rows=("message_id", "size"),
            avg_final_score=("final_score", "mean"),
            hard_cluster_rows=("hard_bot_cluster_flag", "sum"),
            hard_repeat_rows=("hard_same_text_repeat_flag", "sum"),
        )
        .reset_index()
    )
    grouped = grouped[grouped["rows"] >= min_rows].copy()
    grouped["coordination_pressure"] = grouped["hard_cluster_rows"] + grouped["hard_repeat_rows"]
    return grouped.sort_values(["avg_final_score", "coordination_pressure", "rows"], ascending=False)


## Run Selection

The preset chooses the scale and intent of the run:

- `model_test`: lightweight validation on a limited slice of the dataset
- `full_build`: full artifact build over the configured dataset
- `demo`: smallest presentation-oriented preset with semantic scoring disabled

The execution mode chooses the action:

- `build`: create artifacts from scratch (recommended on a fresh machine)
- `rescore`: reuse the existing cleaned store and rewrite scores
- `validate_only`: only validate and inspect existing artifacts


In [ ]:
print({
    "active_preset": ACTIVE_PRESET,
    "execution_mode": EXECUTION_MODE,
    "input_parquet": ACTIVE_CONFIG["paths"]["input_parquet"],
    "batch_sqlite_db": ACTIVE_CONFIG["paths"]["batch_sqlite_db"],
    "sample_n_rows": ACTIVE_CONFIG["runtime"]["sample_n_rows"],
    "max_batches": ACTIVE_CONFIG["runtime"]["max_batches"],
    "semantic_enabled": ACTIVE_CONFIG["semantic_adapter"]["enabled"],
})


## Execute And Validate

This cell is the single execution gate for the notebook.

It always finishes with strict artifact validation, so every later analysis cell reads from a known-good state.


In [ ]:
if EXECUTION_MODE == "build":
    result = run_formula_pipeline_two_pass(ACTIVE_CONFIG)
elif EXECUTION_MODE == "rescore":
    result = run_rescore_from_existing_store(ACTIVE_CONFIG)
elif EXECUTION_MODE == "validate_only":
    result = {}
else:
    raise ValueError(f"Unsupported EXECUTION_MODE: {EXECUTION_MODE}")

result = attach_runtime_paths(result, ACTIVE_CONFIG)
result = assert_artifacts_ready(result, ACTIVE_CONFIG)
manifest = result["manifest"]

print({
    "pipeline_version": manifest["pipeline_version"],
    "store_schema_version": manifest["store_schema_version"],
    "score_schema_version": manifest["score_schema_version"],
    "sqlite_path": result["paths"]["sqlite_db"],
    "scored_messages_path": result["paths"]["scored_messages_parquet"],
})


## Artifact Summary

This summary confirms which artifact set the notebook is reading from and which schema versions are active.


In [ ]:
pd.DataFrame(
    [
        {"field": "pipeline_version", "value": manifest["pipeline_version"]},
        {"field": "store_schema_version", "value": manifest["store_schema_version"]},
        {"field": "score_schema_version", "value": manifest["score_schema_version"]},
        {"field": "created_at", "value": manifest["created_at"]},
        {"field": "last_rescore_at", "value": manifest.get("last_rescore_at")},
        {"field": "sqlite_path", "value": result["paths"]["sqlite_db"]},
        {"field": "author_scores_path", "value": result["paths"]["author_scores_parquet"]},
        {"field": "scored_messages_path", "value": result["paths"]["scored_messages_parquet"]},
    ]
)


## Score Distribution

The `final_score` is a ranked manipulation-risk signal, not a certainty claim.
This section gives the global distribution and a compact band summary.


In [ ]:
score_dist = load_scored_messages(
    ACTIVE_CONFIG,
    columns=["final_score", "behavioral_score", "roberta_score"],
)

score_bands = summarize_score_bands(score_dist[["final_score"]].assign(final_score=score_dist["final_score"]))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(score_dist["final_score"].dropna(), bins=40, color="#2563eb", edgecolor="white")
ax.set_title("Final Score Distribution")
ax.set_xlabel("final_score")
ax.set_ylabel("message count")
fig.tight_layout()
plt.show()

score_bands


## Author-Level Risk Checks

Behavioral evidence is the main source of power in this model.
These views focus on author-level pressure, not just suspicious text.


In [ ]:
author_scores = load_author_scores(ACTIVE_CONFIG)

author_scores.sort_values("author_score", ascending=False).head(15)[[
    "author_hash",
    "author_score",
    "author_hard_hourly_flag",
    "max_posts_one_hour",
    "language_nunique",
    "theme_nunique",
    "median_interpost_sec",
]]


In [ ]:
fig = plot_hourly_distribution(author_scores)
plt.show(fig)

fig = plot_hourly_penalty_curve(author_scores, ACTIVE_CONFIG)
plt.show(fig)


## Repeat And Cluster Diagnostics

This block shows message-level cases where repeated content and cluster pressure are visibly strong.


In [ ]:
scored_preview = load_scored_messages(
    ACTIVE_CONFIG,
    columns=[
        "author_hash",
        "author_type",
        "normalized_text",
        "same_text_repeat_count",
        "same_text_unique_author_count",
        "same_text_time_window_count",
        "author_score",
        "message_score",
        "behavioral_score",
        "roberta_score",
        "final_score",
        "hard_bot_cluster_flag",
        "hard_same_text_repeat_flag",
    ],
)

scored_preview.loc[
    scored_preview["hard_bot_cluster_flag"].eq(1) | scored_preview["hard_same_text_repeat_flag"].eq(1)
].sort_values("final_score", ascending=False).head(15)


## Compact Manipulation Map

This section highlights where manipulation risk concentrates across language and platform.
It is intentionally compact rather than a full dashboard.


In [ ]:
manipulation_map = build_compact_manipulation_map(ACTIVE_CONFIG, min_rows=25)

fig, ax = plt.subplots(figsize=(10, 5))
plot_frame = manipulation_map.head(10).copy()
plot_frame["label"] = plot_frame["language"].fillna("unknown") + " | " + plot_frame["registered_domain"].fillna("unknown")
ax.barh(plot_frame["label"], plot_frame["avg_final_score"], color="#dc2626")
ax.invert_yaxis()
ax.set_title("Top Language/Platform Risk Clusters")
ax.set_xlabel("average final_score")
fig.tight_layout()
plt.show()

manipulation_map.head(15)


## Single-Message Inference Demo

This example is intentionally simple.

A suspicious-looking message is **not** proof of automation by itself. The score becomes more meaningful when the message is evaluated together with existing author and cluster context.


In [ ]:
single_message = prepare_single_message_input(
    message_text="BUY IT NOW!!! SALE!!! #promo #deal",
    language="en",
    url="https://x.com/example",
    date="2024-01-01T00:00:00Z",
    author_hash=None,
    english_keywords="sale, promo, deal",
    primary_theme="promotion",
)

score_single_message(single_message, result, ACTIVE_CONFIG)


## Closing Interpretation

Use this notebook to tell the correct story about the project:

- this is not a perfect bot detector
- text-only evidence is weak
- behavioral patterns are the strongest signal source
- semantic scoring is a supporting feature
- the final score is an explainable manipulation-risk estimate

That framing is both more honest and more technically defensible.
